# UNIVERSIDAD INTERNACIONAL DE VALENCIA
## MIAR - ALGORITMOS DE OPTIMIZACIÓN

**Nombre:** Graciela Elizabeth Medina Topanta

**GoogleColab:** https://drive.google.com/drive/folders/1RTAVTssCcB-UHXLEN4JlCDt6_2Hpm6yX?usp=drive_link

**Github:** https://github.com/gemedinat/MIAR_ALGORITMOS_DE_OPTIMIZACION.git

In [1]:
#Modulo de llamadas http para descargar ficheros
!pip install requests

#Libreria del problema TSP: http://elib.zib.de/pub/mp-testdata/tsp/tsplib/tsplib.html
!pip install tsplib95

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 480.2 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 480.2 kB/s eta 0:00:04
   --------------- ------------------------ 0.8/2.0 MB 496.4 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 496.4 kB/s eta 0:00:03
   --------------- ------------------------ 0.8/2.0 MB 496.4 kB/s eta 0:00:03
   -------------------- ------------------- 1.0/2.0 MB 514.1 kB/s eta 0:00:02
   -------------------- ------------------- 1.0/2.0 MB 514.1 kB/s eta 0:00:02
   ------------------------- -----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
causalts 0.26.0 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
dowhy 0.14 requires networkx>=3.3; python_version >= "3.10", but you have networkx 2.8.8 which is incompatible.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.


In [1]:
import tsplib95
import random
from math import e
import urllib.request

In [2]:
# DATOS DEL PROBLEMA
import os
import urllib.request
import tsplib95

file = "swiss42.tsp"
# Descarga condicional desde mirror estable (antes: http de Heidelberg + gzip -d,
# que falla con el servidor caído o al re-ejecutar la celda)
if not os.path.exists(file):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/mastqe/tsplib/refs/heads/master/swiss42.tsp",
        file
    )

problem = tsplib95.load(file)

# Nodos
Nodos = list(problem.get_nodes())

# Devuelve la distancia entre dos nodos
def distancia(a, b, problema):
    return problema.get_weight(a, b)

# Devuelve la distancia total de una trayectoria/solución (lista de nodos)
def distancia_total(solucion, problema):
    total = 0                       # antes se llamaba igual que la función (sombreado)
    for i in range(len(solucion) - 1):
        total += distancia(solucion[i], solucion[i + 1], problema)
    # Arista de retorno al nodo inicial para cerrar el ciclo
    return total + distancia(solucion[-1], solucion[0], problema)


## Algoritmo de colonia de hormigas

La función Add_Nodo selecciona al azar un nodo con probabilidad uniforme.
Para ser mas eficiente debería seleccionar el próximo nodo siguiendo la probabilidad correspondiente a la ecuación:

$p^k_{ij}(t) = \frac{[\tau_{ij}(t)]^\alpha[\nu_{ij}]^\beta}{\sum_{l\in J^k_i} [\tau_{il}(t)]^\alpha[\nu_{il}]^\beta}$, si $j \in J^k_i$

$p^k_{ij}(t) = 0$, si $j \notin J^k_i$

In [4]:
def Add_Nodo(problem, H ,T ) :
  #Mejora:Establecer una funcion de probabilidad para
  # añadir un nuevo nodo dependiendo de los nodos mas cercanos y de las feromonas depositadas
  Nodos = list(problem.get_nodes())
  return random.choice(   list(set(range(1,len(Nodos))) - set(H) )  )


def Incrementa_Feromona(problem, T, H ) :
  #Incrementa segun la calidad de la solución. Añadir una cantidad inversamente proporcional a la distancia total
  for i in range(len(H)-1):
    T[H[i]][H[i+1]] += 1000/distancia_total(H, problem)
  return T

def Evaporar_Feromonas(T ):
  #Evapora 0.3 el valor de la feromona, sin que baje de 1
  #Mejora:Podemos elegir diferentes funciones de evaporación dependiendo de la cantidad actual y de la suma total de feromonas depositadas,...
  T = [[ max(T[i][j] - 0.3 , 1) for i in range(len(Nodos)) ] for j in range(len(Nodos))]
  return T

In [5]:
def hormigas(problem, N) :
  #problem = datos del problema
  #N = Número de agentes(hormigas)

  #Nodos
  Nodos = list(problem.get_nodes())
  #Aristas
  Aristas = list(problem.get_edges())

  #Inicializa las aristas con una cantidad inicial de feromonas:1
  #Mejora: inicializar con valores diferentes dependiendo diferentes criterios
  T = [[ 1 for _ in range(len(Nodos)) ] for _ in range(len(Nodos))]

  #Se generan los agentes(hormigas) que serán estructuras de caminos desde 0
  Hormiga = [[0] for _ in range(N)]

  #Recorre cada agente construyendo la solución
  for h in range(N) :
    #Para cada agente se construye un camino
    for i in range(len(Nodos)-1) :

      #Elige el siguiente nodo
      Nuevo_Nodo = Add_Nodo(problem, Hormiga[h] ,T )
      Hormiga[h].append(Nuevo_Nodo)

    #Incrementa feromonas en esa arista
    T = Incrementa_Feromona(problem, T, Hormiga[h] )
    #print("Feromonas(1)", T)

    #Evapora Feromonas
    T = Evaporar_Feromonas(T)
    #print("Feromonas(2)", T)

    #Seleccionamos el mejor agente
  mejor_solucion = []
  mejor_distancia = 10e100
  for h in range(N) :
    distancia_actual = distancia_total(Hormiga[h], problem)
    if distancia_actual < mejor_distancia:
      mejor_solucion = Hormiga[h]
      mejor_distancia =distancia_actual


  print(mejor_solucion)
  print(mejor_distancia)


hormigas(problem, 1000)

[0, 30, 38, 3, 41, 11, 20, 34, 27, 6, 5, 1, 31, 29, 36, 15, 2, 21, 9, 28, 40, 13, 4, 33, 35, 17, 7, 32, 14, 24, 23, 22, 8, 39, 18, 10, 25, 26, 12, 16, 37, 19]
3850
